# EgoCom -> ECMC 格式预处理 Notebook

这个 notebook 用于把 EgoCom 的 `ground_truth_transcriptions.csv` 和 `egocom_pretrained_features` 转成当前 ECMC 代码更容易使用的调试数据。

当前目标是先跑通一个小闭环：

1. 按 `speaker turn` 切 transcript。
2. 根据 `video_info.csv` 把 `conversation_id + speaker_id` 映射到 `video_id`。
3. 从 `egocom_features_history_4sec.csv.gz` 抽取对应 `clip_id` 的 video/audio 特征。
4. 导出 ECMC 风格的 `train.csv` 和 `.npy` 特征。
5. 检查 `.npy` shape。

注意：这里先用占位弱标签，后续再接 LLM/API 生成 `emotion`、`cognition`、`emotion_bin`、`cognition_bin`。

In [17]:
from pathlib import Path
import csv
import gzip
import math
import pickle
import re
from collections import defaultdict

import numpy as np
import pandas as pd

# Windows 路径建议用原始字符串 r"..."，避免反斜杠转义问题。
EGO_ROOT = Path(r"F:/egocom")
FEATURE_ROOT = EGO_ROOT / "egocom_pretrained_features.tar" / "egocom_pretrained_features"
TRANSCRIPT_CSV = EGO_ROOT / "ground_truth_transcriptions.csv"
VIDEO_INFO_CSV = EGO_ROOT / "video_info.csv"
FEATURE_CSV_GZ = FEATURE_ROOT / "features_by_history" / "egocom_features_history_4sec.csv.gz"
COLUMN_NAMES_P = FEATURE_ROOT / "feature_column_names.p"

# 输出到项目 my_text 下，避免污染原始 ECMC 根目录。
OUT_ROOT = Path(r"D:/py_code/ECMC/my_text/egocom_ecmc_debug")
OUT_AUDIO_DIR = OUT_ROOT / "MMDA" / "split_audio_f"
OUT_VIDEO_DIR = OUT_ROOT / "MMDA" / "split_video_f"
OUT_TURNS_CSV = OUT_ROOT / "speaker_turns_debug.csv"
OUT_TRAIN_CSV = OUT_ROOT / "train.csv"

# 轻薄本调试参数：先小样本跑通。
MAX_VIDEO_ROWS = 50       # 只取 video_info 里前几个 train video 行
MAX_TURNS = 2000           # 最多导出多少个 speaker turn
MAX_GAP = 1.0            # 同一 speaker 相邻词间隔超过该值则切开，单位秒
MIN_DURATION = 2.0       # 太短的 turn 删除
MIN_WORDS = 3            # 词数太少的 turn 删除
MAX_DURATION = 30.0      # 太长的 turn 强制截断/切开
CHUNKSIZE = 2000         # 流式读取大 csv.gz 的 chunk 大小

for p in [OUT_ROOT, OUT_AUDIO_DIR, OUT_VIDEO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("EGO_ROOT:", EGO_ROOT)
print("FEATURE_ROOT exists:", FEATURE_ROOT.exists())
print("TRANSCRIPT_CSV exists:", TRANSCRIPT_CSV.exists())
print("VIDEO_INFO_CSV exists:", VIDEO_INFO_CSV.exists())
print("FEATURE_CSV_GZ exists:", FEATURE_CSV_GZ.exists())
print("OUT_ROOT:", OUT_ROOT)

EGO_ROOT: F:\egocom
FEATURE_ROOT exists: True
TRANSCRIPT_CSV exists: True
VIDEO_INFO_CSV exists: True
FEATURE_CSV_GZ exists: True
OUT_ROOT: D:\py_code\ECMC\my_text\egocom_ecmc_debug


## 1. 检查特征列

EgoCom 的预提取特征不是 ECMC 默认的 768 维：当前本地文件中 video 是 2048 维，audio 是 512 维，text 是 300 维。后续训练时建议在模型里加 adapter：`Linear(512, 768)` 和 `Linear(2048, 768)`。

In [18]:
with COLUMN_NAMES_P.open("rb") as f:
    feature_columns = list(pickle.load(f))

video_cols = [c for c in feature_columns if str(c).startswith("videofeat_")]
audio_cols = [c for c in feature_columns if str(c).startswith("voxaudiofeat_")]
text_cols = [c for c in feature_columns if str(c).startswith("textfeat_")]
meta_cols = ["video_id", "clip_id", "video_speaker_id", "is_speaking", "multiclass_speaker_label"]

print("total columns:", len(feature_columns))
print("video dim:", len(video_cols), video_cols[:3], video_cols[-3:])
print("audio dim:", len(audio_cols), audio_cols[:3], audio_cols[-3:])
print("text dim:", len(text_cols), text_cols[:3], text_cols[-3:])
print("meta cols:", meta_cols)

total columns: 2865
video dim: 2048 ['videofeat_0', 'videofeat_1', 'videofeat_2'] ['videofeat_2045', 'videofeat_2046', 'videofeat_2047']
audio dim: 512 ['voxaudiofeat_0', 'voxaudiofeat_1', 'voxaudiofeat_2'] ['voxaudiofeat_509', 'voxaudiofeat_510', 'voxaudiofeat_511']
text dim: 300 ['textfeat_0', 'textfeat_1', 'textfeat_2'] ['textfeat_297', 'textfeat_298', 'textfeat_299']
meta cols: ['video_id', 'clip_id', 'video_speaker_id', 'is_speaking', 'multiclass_speaker_label']


## 2. 构造 speaker turns

规则：同一个 `conversation_id` 内，按 `startTime` 排序；连续同一个 `speaker_id` 合并；如果换 speaker、相邻词间隔太大、或片段太长，就切成新的 turn。

In [19]:
def clean_text(words):
    text = " ".join(str(w).strip() for w in words if str(w).strip())
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text


def safe_id(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def flush_turn(turn, rows, min_duration=2.0, min_words=3):
    if not turn:
        return
    duration = float(turn["end"] - turn["start"])
    word_count = len(turn["words"])
    content = clean_text(turn["words"])
    if duration < min_duration or word_count < min_words or not content:
        return

    start_ms = int(round(turn["start"] * 1000))
    end_ms = int(round(turn["end"] * 1000))
    conversation_id = safe_id(turn["conversation_id"])
    speaker_id = safe_id(turn["speaker_id"])

    rows.append({
        "id": f"{conversation_id}_spk{speaker_id}_{start_ms}_{end_ms}",
        "conversation_id": turn["conversation_id"],
        "speaker_id": int(turn["speaker_id"]),
        "start": float(turn["start"]),
        "end": float(turn["end"]),
        "duration": duration,
        "word_count": word_count,
        "content": content,
    })


def build_speaker_turns(transcript_csv, video_info_csv, max_video_rows=3):
    video_info = pd.read_csv(video_info_csv)
    # 先取官方 train split 的前几个 video 行，用于轻量调试。
    train_mask = video_info["train"].astype(str).str.lower().eq("true")
    train_video_info = video_info[train_mask].head(max_video_rows).copy()
    wanted_pairs = set(zip(train_video_info["conversation_id"], train_video_info["video_speaker_id"].astype(int)))
    print("debug video rows:", len(train_video_info))
    print(train_video_info[["video_id", "conversation_id", "video_speaker_id", "duration_seconds", "train", "val", "test"]])

    df = pd.read_csv(transcript_csv)
    df = df.dropna(subset=["conversation_id", "speaker_id", "startTime", "endTime", "word"])
    df["speaker_id"] = pd.to_numeric(df["speaker_id"], errors="coerce").astype("Int64")
    df["startTime"] = pd.to_numeric(df["startTime"], errors="coerce")
    df["endTime"] = pd.to_numeric(df["endTime"], errors="coerce")
    df = df.dropna(subset=["speaker_id", "startTime", "endTime"])
    df["speaker_id"] = df["speaker_id"].astype(int)
    df = df[df.apply(lambda r: (r["conversation_id"], int(r["speaker_id"])) in wanted_pairs, axis=1)]
    df = df.sort_values(["conversation_id", "startTime", "endTime"]).reset_index(drop=True)

    rows = []
    for conversation_id, group in df.groupby("conversation_id", sort=False):
        current = None
        for row in group.itertuples(index=False):
            speaker_id = int(row.speaker_id)
            start = float(row.startTime)
            end = float(row.endTime)
            word = str(row.word).strip()
            if not word:
                continue

            if current is None:
                current = {
                    "conversation_id": conversation_id,
                    "speaker_id": speaker_id,
                    "start": start,
                    "end": end,
                    "words": [word],
                }
                continue

            gap = start - current["end"]
            duration_if_added = end - current["start"]
            should_split = speaker_id != current["speaker_id"] or gap > MAX_GAP or duration_if_added > MAX_DURATION
            if should_split:
                flush_turn(current, rows, MIN_DURATION, MIN_WORDS)
                current = {
                    "conversation_id": conversation_id,
                    "speaker_id": speaker_id,
                    "start": start,
                    "end": end,
                    "words": [word],
                }
            else:
                current["end"] = max(current["end"], end)
                current["words"].append(word)
        flush_turn(current, rows, MIN_DURATION, MIN_WORDS)

    turns = pd.DataFrame(rows)
    turns = turns.merge(
        video_info[["video_id", "conversation_id", "video_speaker_id", "train", "val", "test"]],
        left_on=["conversation_id", "speaker_id"],
        right_on=["conversation_id", "video_speaker_id"],
        how="left",
    )
    turns = turns.dropna(subset=["video_id"]).copy()
    turns["video_id"] = turns["video_id"].astype(int)
    turns["start_clip"] = np.floor(turns["start"]).astype(int)
    turns["end_clip"] = np.ceil(turns["end"]).astype(int)
    turns = turns.head(MAX_TURNS).reset_index(drop=True)
    return turns


turns = build_speaker_turns(TRANSCRIPT_CSV, VIDEO_INFO_CSV, MAX_VIDEO_ROWS)
turns.to_csv(OUT_TURNS_CSV, index=False, encoding="utf-8-sig")
print("speaker turns:", len(turns))
print("saved:", OUT_TURNS_CSV)
turns.head(10)

debug video rows: 50
    video_id      conversation_id  video_speaker_id  duration_seconds  train  \
0          1  day_1__con_1__part1                 1               275   True   
1          2  day_1__con_1__part2                 1               295   True   
2          3  day_1__con_1__part3                 1               295   True   
3          4  day_1__con_1__part4                 1               295   True   
4          5  day_1__con_1__part5                 1                83   True   
5          6  day_1__con_1__part1                 2               275   True   
6          7  day_1__con_1__part2                 2               295   True   
7          8  day_1__con_1__part3                 2               295   True   
8          9  day_1__con_1__part4                 2               295   True   
9         10  day_1__con_1__part5                 2                83   True   
10        11  day_1__con_1__part1                 3               275   True   
11        12  day_1

,id,conversation_id,speaker_id,start,end,duration,word_count,content,video_id,video_speaker_id,train,val,test,start_clip,end_clip
0,day_1__con_1__part1_spk1_90_7640,day_1__con_1__part1,1,0.09,7.64,7.55,39,"Okay. So, I have some topics in my hand and I ...",1,1,True,False,False,0,8
1,day_1__con_1__part1_spk3_20370_22610,day_1__con_1__part1,3,20.37,22.61,2.24,19,"Curtis, why didn ' t you wear pants today? The...",11,3,True,False,False,20,23
2,day_1__con_1__part1_spk2_25920_28010,day_1__con_1__part1,2,25.92,28.01,2.09,11,"The office is always so cold, though. Like -",6,2,True,False,False,25,29
3,day_1__con_1__part1_spk2_28220_32770,day_1__con_1__part1,2,28.22,32.77,4.55,36,"... I go outside and I ' m like, "" I wish I we...",6,2,True,False,False,28,33
4,day_1__con_1__part1_spk1_38060_40900,day_1__con_1__part1,1,38.06,40.90,2.84,16,"Um, what else? What is a - what ' s a second t...",1,1,True,False,False,38,41
5,day_1__con_1__part1_spk2_69920_74460,day_1__con_1__part1,2,69.92,74.46,4.54,47,( laughs ) [ inaudible 0: 1: 10 ] I was gonna ...,6,2,True,False,False,69,75
6,day_1__con_1__part1_spk2_80060_82220,day_1__con_1__part1,2,80.06,82.22,2.16,15,I ' ve actually never been in here before. Can...,6,2,True,False,False,80,83
7,day_1__con_1__part1_spk1_84980_90580,day_1__con_1__part1,1,84.98,90.58,5.60,57,"Um, we can a little bit, but, uh, keep - keep ...",1,1,True,False,False,84,91
8,day_1__con_1__part1_spk2_111500_115700,day_1__con_1__part1,2,111.50,115.70,4.20,11,( laughs ) And one bottle of rum. Exciting.,6,2,True,False,False,111,116
9,day_1__con_1__part1_spk1_116480_119350,day_1__con_1__part1,1,116.48,119.35,2.87,11,This is TRUE. Cooking - for cooking purposes o...,1,1,True,False,False,116,120


## 3. 从 EgoCom 大表中抽取 turn 对应的 audio/video 特征

这里不把整份 `csv.gz` 解压到硬盘，而是用 `pandas.read_csv(..., chunksize=...)` 流式扫描，只保留 debug turns 需要的 `(video_id, clip_id)`。

In [20]:
def build_target_clip_map(turns_df):
    target_pairs = set()
    turn_to_pairs = {}
    for row in turns_df.itertuples(index=False):
        pairs = []
        for clip_id in range(int(row.start_clip), int(row.end_clip) + 1):
            pair = (int(row.video_id), int(clip_id))
            pairs.append(pair)
            target_pairs.add(pair)
        turn_to_pairs[row.id] = pairs
    return target_pairs, turn_to_pairs


def load_needed_feature_rows(feature_csv_gz, target_pairs):
    usecols = meta_cols + video_cols + audio_cols
    needed = {}
    total_seen = 0
    total_kept = 0

    for chunk_idx, chunk in enumerate(pd.read_csv(feature_csv_gz, usecols=usecols, chunksize=CHUNKSIZE)):
        chunk["video_id"] = chunk["video_id"].astype(int)
        chunk["clip_id"] = chunk["clip_id"].astype(int)
        mask = [(int(v), int(c)) in target_pairs for v, c in zip(chunk["video_id"], chunk["clip_id"])]
        kept = chunk.loc[mask]
        for row in kept.itertuples(index=False):
            key = (int(row.video_id), int(row.clip_id))
            video = np.array([getattr(row, c) for c in video_cols], dtype=np.float32)
            audio = np.array([getattr(row, c) for c in audio_cols], dtype=np.float32)
            needed[key] = {"video": video, "audio": audio}
        total_seen += len(chunk)
        total_kept += len(kept)
        if chunk_idx % 50 == 0:
            print(f"chunk={chunk_idx}, seen={total_seen}, kept={total_kept}, needed={len(needed)}/{len(target_pairs)}")
        if len(needed) >= len(target_pairs):
            break
    print(f"finished: seen={total_seen}, kept={total_kept}, loaded={len(needed)}/{len(target_pairs)}")
    return needed


target_pairs, turn_to_pairs = build_target_clip_map(turns)
print("target clip pairs:", len(target_pairs))
feature_rows = load_needed_feature_rows(FEATURE_CSV_GZ, target_pairs)

target clip pairs: 3155
chunk=0, seen=2000, kept=506, needed=506/3155
chunk=50, seen=102000, kept=3071, needed=3071/3155
finished: seen=136431, kept=3071, loaded=3071/3155


## 4. 导出 ECMC debug 数据

当前先写占位弱标签，保证数据管线能跑通。后续要把这部分替换为 LLM/API 生成的结果。

In [21]:
def placeholder_emotion_label(text):
    # 占位规则：只保证格式正确，不代表真实标注。
    lower = str(text).lower()
    negative_words = ["hard", "scared", "terrifying", "bad", "hate", "worst", "can't", "cannot"]
    positive_words = ["good", "great", "happy", "awesome", "wonderful", "nice", "love"]
    if any(w in lower for w in negative_words):
        return -1
    if any(w in lower for w in positive_words):
        return 1
    return 0


def export_ecmc_debug_dataset(turns_df, feature_rows, turn_to_pairs):
    csv_rows = []
    skipped = 0
    for row in turns_df.itertuples(index=False):
        pairs = turn_to_pairs[row.id]
        audio_seq = []
        video_seq = []
        for pair in pairs:
            item = feature_rows.get(pair)
            if item is None:
                continue
            audio_seq.append(item["audio"])
            video_seq.append(item["video"])

        expected_len = len(pairs)
        coverage = len(audio_seq) / max(expected_len, 1)
        # EgoCom 官方窗口特征在视频尾部可能缺少若干 clip。覆盖率太低的 turn 直接跳过。
        if not audio_seq or not video_seq or coverage < 0.8:
            skipped += 1
            continue

        audio_arr = np.stack(audio_seq).astype(np.float32)  # [T, 512]
        video_arr = np.stack(video_seq).astype(np.float32)  # [T, 2048]
        np.save(OUT_AUDIO_DIR / f"{row.id}.npy", audio_arr)
        np.save(OUT_VIDEO_DIR / f"{row.id}.npy", video_arr)

        emo_bin = placeholder_emotion_label(row.content)
        csv_rows.append({
            "id": row.id,
            "content": row.content,
            "emotion_bin": emo_bin,
            "cognition_bin": "[0, 0, 0, 0]",
            "emotion": "占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。",
            "cognition": "占位认知描述：该片段的真实 cognition-style caption 需要后续由 LLM/人工校验生成。",
            "conversation_id": row.conversation_id,
            "speaker_id": row.speaker_id,
            "video_id": row.video_id,
            "start": row.start,
            "end": row.end,
            "feature_len": len(audio_seq),
            "expected_feature_len": expected_len,
            "feature_coverage": round(coverage, 4),
        })

    out_df = pd.DataFrame(csv_rows)
    out_df.to_csv(OUT_TRAIN_CSV, index=False, encoding="utf-8-sig")
    print("exported rows:", len(out_df), "skipped:", skipped)
    print("saved train csv:", OUT_TRAIN_CSV)
    print("audio dir:", OUT_AUDIO_DIR)
    print("video dir:", OUT_VIDEO_DIR)
    return out_df


# 如果从这一格单独开始运行，先尝试恢复前面保存/计算过的中间变量。
if "turns" not in globals():
    if OUT_TURNS_CSV.exists():
        turns = pd.read_csv(OUT_TURNS_CSV, encoding="utf-8-sig")
        print("已从文件恢复 turns:", OUT_TURNS_CSV, "rows=", len(turns))
    else:
        raise RuntimeError("turns 未定义，且找不到 speaker_turns_debug.csv。请先运行『构造 speaker turns』那一格。")

if "turn_to_pairs" not in globals() or "target_pairs" not in globals():
    target_pairs, turn_to_pairs = build_target_clip_map(turns)
    print("已重建 turn_to_pairs，target clip pairs:", len(target_pairs))

if "feature_rows" not in globals():
    print("feature_rows 未定义，开始重新从 EgoCom 大表中读取需要的特征。")
    feature_rows = load_needed_feature_rows(FEATURE_CSV_GZ, target_pairs)

train_df = export_ecmc_debug_dataset(turns, feature_rows, turn_to_pairs)
train_df.head()

exported rows: 441 skipped: 23
saved train csv: D:\py_code\ECMC\my_text\egocom_ecmc_debug\train.csv
audio dir: D:\py_code\ECMC\my_text\egocom_ecmc_debug\MMDA\split_audio_f
video dir: D:\py_code\ECMC\my_text\egocom_ecmc_debug\MMDA\split_video_f


,id,content,emotion_bin,cognition_bin,emotion,cognition,conversation_id,speaker_id,video_id,start,end,feature_len,expected_feature_len,feature_coverage
0,day_1__con_1__part1_spk1_90_7640,"Okay. So, I have some topics in my hand and I ...",0,"[0, 0, 0, 0]",占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。,占位认知描述：该片段的真实 cognition-style caption 需要后续由 LL...,day_1__con_1__part1,1,1,0.09,7.64,9,9,1.0
1,day_1__con_1__part1_spk3_20370_22610,"Curtis, why didn ' t you wear pants today? The...",0,"[0, 0, 0, 0]",占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。,占位认知描述：该片段的真实 cognition-style caption 需要后续由 LL...,day_1__con_1__part1,3,11,20.37,22.61,4,4,1.0
2,day_1__con_1__part1_spk2_25920_28010,"The office is always so cold, though. Like -",0,"[0, 0, 0, 0]",占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。,占位认知描述：该片段的真实 cognition-style caption 需要后续由 LL...,day_1__con_1__part1,2,6,25.92,28.01,5,5,1.0
3,day_1__con_1__part1_spk2_28220_32770,"... I go outside and I ' m like, "" I wish I we...",0,"[0, 0, 0, 0]",占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。,占位认知描述：该片段的真实 cognition-style caption 需要后续由 LL...,day_1__con_1__part1,2,6,28.22,32.77,6,6,1.0
4,day_1__con_1__part1_spk1_38060_40900,"Um, what else? What is a - what ' s a second t...",0,"[0, 0, 0, 0]",占位情绪描述：该片段的真实 emotion caption 需要后续由 LLM/人工校验生成。,占位认知描述：该片段的真实 cognition-style caption 需要后续由 LL...,day_1__con_1__part1,1,1,38.06,40.90,4,4,1.0


## 5. 检查导出的 `.npy` shape

预期：

- audio: `[T, 512]`
- video: `[T, 2048]`

这一步只检查数据是否导出成功。训练 ECMC 前，需要给模型加 adapter 或修改输入维度。

In [22]:
audio_files = sorted(OUT_AUDIO_DIR.glob("*.npy"))
video_files = sorted(OUT_VIDEO_DIR.glob("*.npy"))
print("audio npy count:", len(audio_files))
print("video npy count:", len(video_files))

for a_path, v_path in list(zip(audio_files, video_files))[:10]:
    a = np.load(a_path)
    v = np.load(v_path)
    print(a_path.name, "audio", a.shape, "video", v.shape)

audio npy count: 470
video npy count: 470
day_1__con_1__part1_spk1_116480_119350.npy audio (5, 512) video (5, 2048)
day_1__con_1__part1_spk1_130710_135260.npy audio (7, 512) video (7, 2048)
day_1__con_1__part1_spk1_136290_138730.npy audio (4, 512) video (4, 2048)
day_1__con_1__part1_spk1_161510_167430.npy audio (8, 512) video (8, 2048)
day_1__con_1__part1_spk1_163820_167430.npy audio (6, 512) video (6, 2048)
day_1__con_1__part1_spk1_206030_208910.npy audio (4, 512) video (4, 2048)
day_1__con_1__part1_spk1_220430_224630.npy audio (6, 512) video (6, 2048)
day_1__con_1__part1_spk1_221900_224630.npy audio (5, 512) video (5, 2048)
day_1__con_1__part1_spk1_243760_246500.npy audio (5, 512) video (5, 2048)
day_1__con_1__part1_spk1_243760_248910.npy audio (7, 512) video (7, 2048)


## 6. 后续接 LLM 弱标注的位置

当前 `train.csv` 里的 `emotion/cognition` 是占位文本。后续建议新增一列或直接覆盖：

- `emotion_bin`: `-1/0/1`，表示 negative / neutral / positive。
- `cognition_bin`: 四维列表字符串，如 `[0, 1, 0, 0]`。
- `emotion`: 情绪描述 caption。
- `cognition`: 认知风格 caption。

推荐 LLM prompt 原则：只根据 transcript 生成弱标注，不做临床诊断，不声称使用表情或语音语调。

In [23]:
EMOTION_PROMPT_TEMPLATE = """
You are creating weak emotion annotations for a non-clinical conversation dataset.
Only use the transcript. Do not infer mental health diagnosis.

Transcript:
{content}

Return JSON with exactly these fields:
{{
  "emotion_caption": "one concise sentence",
  "valence": "negative|neutral|positive",
  "confidence": 0.0
}}
""".strip()

COGNITION_PROMPT_TEMPLATE = """
You are creating weak cognition-style annotations for a non-clinical conversation dataset.
Only annotate observable conversational cues. Do not diagnose mental disorders.
If there is no clear evidence, set all four labels to 0.

Definitions:
- orientation_deficit: confusion about person, place, topic, rules, or current task.
- attention_deficit: losing track, ignoring prior context, or drifting off-topic.
- memory_deficit: difficulty recalling recently mentioned information.
- language_disorder: disorganized, incoherent, or clearly impaired expression.

Transcript:
{content}

Return JSON with exactly these fields:
{{
  "cognition_caption": "one concise sentence",
  "cognition_bin": [0, 0, 0, 0],
  "confidence": 0.0
}}
""".strip()

example = train_df.iloc[0]["content"] if len(train_df) else ""
print("Emotion prompt example:\n")
print(EMOTION_PROMPT_TEMPLATE.format(content=example)[:1200])
print("\nCognition prompt example:\n")
print(COGNITION_PROMPT_TEMPLATE.format(content=example)[:1600])

Emotion prompt example:

You are creating weak emotion annotations for a non-clinical conversation dataset.
Only use the transcript. Do not infer mental health diagnosis.

Transcript:
Okay. So, I have some topics in my hand and I ' ll start with, " Name three things that we all have in common right now based on what we ' re wearing. "

Return JSON with exactly these fields:
{
  "emotion_caption": "one concise sentence",
  "valence": "negative|neutral|positive",
  "confidence": 0.0
}

Cognition prompt example:

You are creating weak cognition-style annotations for a non-clinical conversation dataset.
Only annotate observable conversational cues. Do not diagnose mental disorders.
If there is no clear evidence, set all four labels to 0.

Definitions:
- orientation_deficit: confusion about person, place, topic, rules, or current task.
- attention_deficit: losing track, ignoring prior context, or drifting off-topic.
- memory_deficit: difficulty recalling recently mentioned information.
- la